In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

In [2]:
base = "/kaggle/input/competitions/consumer-complaint-classification-challenge/"
train_dataset = "/kaggle/input/competitions/complaint-sense-consumer-complaint-classification-challenge/train_complaints.csv"
test_dataset = "/kaggle/input/competitions/complaint-sense-consumer-complaint-classification-challenge/test_complaints.csv"

train_df = pd.read_csv(train_dataset)
test_df = pd.read_csv(test_dataset)

test_df.head(5)

,ComplaintId,text
0,1001,Hi support — Portal login fails when I try to ...
1,1002,Flagging an issue: Does pausing membership sto...
2,1003,"Hello, Our household's invoice shows a tax amo..."
3,1004,Hi support — International shipment held at cu...
4,1005,Hi support — Fraud alert ignored and three mor...


In [3]:
train_df.head(5)

,ComplaintId,text,Category,FamilyId
0,1,Flagging an issue: Username change broke acces...,account_access,template:account_access:5
1,2,Writing to complain: Blender motor smells like...,product_defect,template:product_defect:6
2,3,"Hello, Courier left perishable groceries in th...",delivery_shipping,template:delivery_shipping:1
3,4,Hi support — Tablet touchscreen registers ghos...,product_defect,template:product_defect:7
4,5,Hi support — Authorized service charged labor ...,warranty_repair,template:warranty_repair:3


In [4]:
train_df.info() #no nuls pretty clean

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ComplaintId  380 non-null    int64 
 1   text         380 non-null    object
 2   Category     380 non-null    object
 3   FamilyId     380 non-null    object
dtypes: int64(1), object(3)
memory usage: 12.0+ KB


In [5]:
# train_df["text"].value_counts()
train_df["FamilyId"].value_counts()

FamilyId
template:account_access:5        5
template:product_defect:6        5
template:delivery_shipping:1     5
template:product_defect:7        5
template:warranty_repair:3       5
                                ..
template:delivery_shipping:5     5
confusable:4                     5
template:fraud_unauthorized:4    5
template:customer_service:0      5
template:general_inquiry:5       5
Name: count, Length: 76, dtype: int64

In [6]:
#### 76 unique family ids and each is tied to 5 rows

In [7]:
train_df["Category"].value_counts()

Category
billing                50
account_access         40
refund_return          40
delivery_shipping      40
fraud_unauthorized     40
product_defect         35
general_inquiry        35
subscription_cancel    35
warranty_repair        35
customer_service       30
Name: count, dtype: int64

In [8]:
#### billing most, custmer service least. the counts are 1 : 4 : 4 : 1

In [9]:
# train_df.describe()
# this doesn't tell us much because most of the data is object, and the complaint id is ordinal

In [10]:
X_train_full = train_df.drop("Category", axis=1)

y_train_full = train_df["Category"].copy()

In [11]:
# creating a stratified validation split, because this is a small dataset and random sampling can cause samplling noise

# from sklearn.model_selection import train_test_split

# # Assuming `X_train_full` and `y_train_full` are your 380 rows of training data & labels
# # We hold out 20% (76 rows) for validation, and use 80% (304 rows) for training
# X_train, X_val, y_train, y_val = train_test_split(
#     X_train_full, 
#     y_train_full, 
#     test_size=0.2, 
#     stratify=y_train_full,   # Stratifies based on the Category
#     random_state=42          # Ensures the split is perfectly reproducible [9]
# )

In [12]:
from sklearn.model_selection import StratifiedGroupKFold

# Never use ComplaintId or FamilyId as model features.
X = train_df["text"]
y = train_df["Category"]
groups = train_df["FamilyId"]

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

train_idx, val_idx = next(cv.split(X, y, groups=groups))

X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

print(X_train.shape, X_val.shape)
print("Shared families:", set(groups.iloc[train_idx]) & set(groups.iloc[val_idx]))

(300,) (80,)
Shared families: set()


In [13]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report

baseline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True
    )),
    ("model", LinearSVC(C=1.0))
])

baseline.fit(X_train, y_train)
val_pred = baseline.predict(X_val)

macro_f1 = f1_score(y_val, val_pred, average="macro")
print(f"Validation Macro F1: {macro_f1:.4f}")
print(classification_report(y_val, val_pred, zero_division=0))

Validation Macro F1: 0.2577
                     precision    recall  f1-score   support

     account_access       0.00      0.00      0.00        10
            billing       0.38      0.33      0.36        15
   customer_service       0.00      0.00      0.00         5
  delivery_shipping       0.20      1.00      0.33         5
 fraud_unauthorized       0.00      0.00      0.00         5
    general_inquiry       0.00      0.00      0.00         5
     product_defect       0.00      0.00      0.00        10
      refund_return       0.33      1.00      0.50         5
subscription_cancel       1.00      0.60      0.75         5
    warranty_repair       1.00      0.47      0.64        15

           accuracy                           0.31        80
          macro avg       0.29      0.34      0.26        80
       weighted avg       0.36      0.31      0.29        80



In [14]:
import torch
print("GPU available:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Enable GPU in Kaggle Settings")

GPU available: True
Tesla T4


In [22]:
!pip install -q transformers datasets accelerate

from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)
from sklearn.metrics import f1_score
import numpy as np

MODEL_NAME = "distilbert-base-uncased"
label_names = sorted(train_df["Category"].unique())
label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for label, i in label2id.items()}

train_data = Dataset.from_dict({
    "text": X_train.tolist(),
    "label": [label2id[label] for label in y_train]
})

val_data = Dataset.from_dict({
    "text": X_val.tolist(),
    "label": [label2id[label] for label in y_val]
})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

train_tokens = train_data.map(tokenize, batched=True)
val_tokens = val_data.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    label2id=label2id,
    id2label=id2label
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "macro_f1": f1_score(labels, predictions, average="macro")
    }

args = TrainingArguments(
    output_dir="./distilbert_complaints",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=10,
    seed=42,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tokens,
    eval_dataset=val_tokens,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()
print(trainer.evaluate())

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, 

Epoch,Training Loss,Validation Loss,Macro F1
1,4.541967,4.563060,0.032000
2,4.303339,4.475183,0.131970
3,3.994375,4.324941,0.220658
4,3.631247,4.196624,0.339885
5,3.303046,4.063087,0.422014
6,2.988511,3.942601,0.428298
7,2.679896,3.857265,0.437169
8,2.479747,3.795015,0.403416
9,2.330289,3.750367,0.424929
10,2.246375,3.733441,0.433074


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 3.8572604656219482, 'eval_macro_f1': 0.4371689629568517, 'eval_runtime': 0.1123, 'eval_samples_per_second': 712.21, 'eval_steps_per_second': 17.805, 'epoch': 10.0}


In [18]:
print("Best metric:", trainer.state.best_metric)
print("Best checkpoint:", trainer.state.best_model_checkpoint)

test_data = Dataset.from_dict({"text": test_df["text"].tolist()})
test_tokens = test_data.map(tokenize, batched=True)

test_logits = trainer.predict(test_tokens).predictions
test_predictions = np.argmax(test_logits, axis=1)

submission = pd.DataFrame({
    "ComplaintId": test_df["ComplaintId"],
    "Category": [id2label[pred] for pred in test_predictions]
})

assert submission.shape == (160, 2)
assert submission.columns.tolist() == ["ComplaintId", "Category"]
assert set(submission["Category"]).issubset(set(label_names))

submission.to_csv("/kaggle/working/submission_distilbert.csv", index=False)
submission.head(), submission["Category"].value_counts()

Best metric: 0.45714285714285713
Best checkpoint: ./distilbert_complaints/checkpoint-80


Map:   0%|          | 0/160 [00:00<?, ? examples/s]

(   ComplaintId            Category
 0         1001      account_access
 1         1002     general_inquiry
 2         1003             billing
 3         1004   delivery_shipping
 4         1005  fraud_unauthorized,
 Category
 refund_return          40
 fraud_unauthorized     34
 subscription_cancel    26
 billing                21
 general_inquiry        15
 product_defect         15
 delivery_shipping       5
 account_access          4
 Name: count, dtype: int64)

In [23]:
import gc
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed
)

MODEL_NAME = "distilbert-base-uncased"
N_SPLITS = 10
EPOCHS = 12
LEARNING_RATE = 2e-5
MAX_LENGTH = 128
SEED = 42

label_names = sorted(train_df["Category"].unique())
label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for label, i in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

def probabilities(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(shifted)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)

# Keep FamilyId only for safe splitting; it never enters the model.
texts = train_df["text"].reset_index(drop=True)
labels = train_df["Category"].reset_index(drop=True)
families = train_df["FamilyId"].reset_index(drop=True)

test_data = Dataset.from_dict({"text": test_df["text"].tolist()})
test_tokens = test_data.map(tokenize, batched=True)

oof_probabilities = np.zeros((len(train_df), len(label_names)))
test_probabilities = np.zeros((len(test_df), len(label_names)))
fold_scores = []

cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

for fold, (train_idx, val_idx) in enumerate(cv.split(texts, labels, families), start=1):
    print(f"\n{'=' * 18} Fold {fold}/{N_SPLITS} {'=' * 18}")

    set_seed(SEED + fold)

    fold_train = Dataset.from_dict({
        "text": texts.iloc[train_idx].tolist(),
        "label": [label2id[x] for x in labels.iloc[train_idx]]
    }).map(tokenize, batched=True)

    fold_val = Dataset.from_dict({
        "text": texts.iloc[val_idx].tolist(),
        "label": [label2id[x] for x in labels.iloc[val_idx]]
    }).map(tokenize, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(label_names),
        label2id=label2id,
        id2label=id2label
    )

    args = TrainingArguments(
        output_dir=f"./cv_distilbert_fold_{fold}",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        eval_strategy="no",
        save_strategy="no",
        logging_strategy="no",
        fp16=torch.cuda.is_available(),
        seed=SEED + fold,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=fold_train,
        data_collator=data_collator
    )

    trainer.train()

    # Out-of-fold validation prediction: each family is evaluated by a model
    # that never trained on that family.
    val_logits = trainer.predict(fold_val).predictions
    val_probs = probabilities(val_logits)
    oof_probabilities[val_idx] = val_probs

    fold_predictions = np.argmax(val_probs, axis=1)
    fold_score = f1_score(
        [label2id[x] for x in labels.iloc[val_idx]],
        fold_predictions,
        average="macro"
    )
    fold_scores.append(fold_score)
    print(f"Fold {fold} macro F1: {fold_score:.4f}")

    # Each fold votes on the test set.
    test_logits = trainer.predict(test_tokens).predictions
    test_probabilities += probabilities(test_logits) / N_SPLITS

    del model, trainer, fold_train, fold_val
    gc.collect()
    torch.cuda.empty_cache()

oof_predictions = np.argmax(oof_probabilities, axis=1)
oof_macro_f1 = f1_score(
    [label2id[x] for x in labels],
    oof_predictions,
    average="macro"
)

print("\nFold scores:", [round(score, 4) for score in fold_scores])
print(f"Mean fold F1: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
print(f"Overall out-of-fold Macro F1: {oof_macro_f1:.4f}")

test_predictions = np.argmax(test_probabilities, axis=1)

submission_ensemble = pd.DataFrame({
    "ComplaintId": test_df["ComplaintId"],
    "Category": [id2label[pred] for pred in test_predictions]
})

assert submission_ensemble.shape == (160, 2)
assert submission_ensemble.columns.tolist() == ["ComplaintId", "Category"]
assert set(submission_ensemble["Category"]).issubset(set(label_names))

submission_ensemble.to_csv(
    "/kaggle/working/submission_distilbert_5fold_ensemble.csv",
    index=False
)

print("\nSaved: /kaggle/working/submission_distilbert_5fold_ensemble.csv")
display(submission_ensemble.head())
display(submission_ensemble["Category"].value_counts())

Map:   0%|          | 0/160 [00:00<?, ? examples/s]


================== Fold 1/10 ==================


Map:   0%|          | 0/340 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Step,Training Loss


Fold 10 macro F1: 0.4680



Fold scores: [0.2798, 0.4384, 0.3158, 0.3904, 0.4084, 0.619, 0.3125, 0.3111, 0.6878, 0.468]
Mean fold F1: 0.4231 ± 0.1299
Overall out-of-fold Macro F1: 0.4942

Saved: /kaggle/working/submission_distilbert_5fold_ensemble.csv


,ComplaintId,Category
0,1001,account_access
1,1002,general_inquiry
2,1003,billing
3,1004,delivery_shipping
4,1005,customer_service


Category
billing               38
warranty_repair       24
refund_return         18
account_access        16
delivery_shipping     16
product_defect        15
general_inquiry       15
fraud_unauthorized    11
customer_service       7
Name: count, dtype: int64